In [1]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import torchvision.transforms as transforms
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/My Drive/diffusion_project" # Replace with your path to the diffusion_project folder
os.chdir(path)
print(f"Current working directory: {os.getcwd()}")
from guided_diffusion.gaussian_diffusion import get_named_beta_schedule

from util.img_utils import clear_color
from util.setup_env import setup_diffusion_environment

setup = setup_diffusion_environment()
model = setup["model"]
device = setup["device"]

num_timesteps = 1000
betas = get_named_beta_schedule("linear", num_diffusion_timesteps=num_timesteps)
alphas = 1.0 - betas
alphas_cumprod = np.cumprod(alphas, axis=0)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

img = Image.open("data/samples/00003.png").convert("RGB")
x_0 = transform(img).to(device).unsqueeze(0)

Mounted at /content/drive
Current working directory: /content/drive/My Drive/diffusion_project


2026-03-14 03:07:26,600 [DPS] >> Device set to cpu.
INFO:DPS:Device set to cpu.


In [2]:
import numpy as np
import torch
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

img = Image.open("data/samples/00003.png").convert("RGB")
x_0 = transform(img).unsqueeze(0).to(device)

In [3]:
def compute_score(x_t, t_index):
    t_tensor = torch.tensor([t_index], device=x_t.device)
    model_output = model(x_t, t_tensor)
    eps_pred, _ = torch.split(model_output, x_t.shape[1], dim=1)
    return -eps_pred / np.sqrt(1.0 - alphas_cumprod[t_index])


def score_jvp(x_t, t_index, v, eps=1e-2):
    with torch.no_grad():
        s_plus = compute_score(x_t + eps * v, t_index)
        s_minus = compute_score(x_t - eps * v, t_index)
    return (s_plus - s_minus) / (2.0 * eps)


def score_vjp(x_t, t_index, v):
    x = x_t.detach().clone().requires_grad_(True)
    score = compute_score(x, t_index)
    scalar = (score * v).sum()
    jTv = torch.autograd.grad(scalar, x, create_graph=False)[0]
    return jTv.detach()


def full_step_vjp(x_t, t_index, v):
    alpha_t = alphas[t_index]
    jTv_score = score_vjp(x_t, t_index, v)
    return (1.0 / np.sqrt(alpha_t)) * (v + (1.0 - alpha_t) * jTv_score)


def make_pixel_delta(shape, channel, row, col, device):
    e_i = torch.zeros(shape, device=device)
    e_i[0, channel, row, col] = 1.0
    return e_i


def power_iteration(x_t, t_index, num_iters=50, use_full_step=True):
    v = torch.randn_like(x_t)
    v = v / v.norm()
    vjp_fn = full_step_vjp if use_full_step else score_vjp

    convergence = []
    for _ in range(num_iters):
        Jv = vjp_fn(x_t, t_index, v)
        eigenvalue = (Jv * v).sum().item()
        convergence.append(eigenvalue)
        norm = Jv.norm().item()
        if norm < 1e-12:
            break
        v = Jv / norm

    return eigenvalue, abs(eigenvalue), convergence

In [4]:
probe_channel = 0
t_probe = 500

torch.manual_seed(0)
z = torch.randn_like(x_0)
alpha_bar = alphas_cumprod[t_probe]
x_t = (np.sqrt(alpha_bar) * x_0 + np.sqrt(1.0 - alpha_bar) * z).to(device)

probe_locations = {
    "Center": (128, 128),
    "Hair Left": (50, 50),
    "Eye region": (115, 110),
    "Hair Right": (30, 200),
}

greens_fns = {}
for name, (row, col) in probe_locations.items():
    e_i = make_pixel_delta(x_0.shape, probe_channel, row, col, device=device)
    g = score_jvp(x_t, t_probe, e_i)
    greens_fns[name] = g[0].cpu().numpy()

In [5]:
channel_names = ["Red", "Green", "Blue"]

n_probes = len(probe_locations)
fig, axes = plt.subplots(n_probes, 4, figsize=(18, 4 * n_probes), constrained_layout=True)
if n_probes == 1:
    axes = axes[np.newaxis, :]

all_vals = np.concatenate([np.abs(g).flatten() for g in greens_fns.values()])
vmax_global = np.percentile(all_vals, 99.5)

for i, (name, (row, col)) in enumerate(probe_locations.items()):
    g = greens_fns[name]

    axes[i, 0].imshow(clear_color(x_t))
    axes[i, 0].plot(col, row, 'r+', markersize=15, markeredgewidth=2)
    axes[i, 0].add_patch(plt.Circle((col, row), 20, color='red', fill=False, linewidth=1.5))
    axes[i, 0].set_title(f"{name} ({row},{col})")
    axes[i, 0].axis("off")

    for c in range(3):
        im = axes[i, c + 1].imshow(g[c], cmap="RdBu_r", vmin=-vmax_global, vmax=vmax_global)
        axes[i, c + 1].plot(col, row, 'k+', markersize=8, markeredgewidth=1)
        axes[i, c + 1].set_title(channel_names[c])
        axes[i, c + 1].axis("off")

fig.colorbar(im, ax=axes[:, -1].tolist(), shrink=0.6, label=r"$J_s e_i$")
fig.suptitle(f"Score Jacobian impulse responses at different locations (t={t_probe})", y=1.01)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [6]:
def ddpm_step(x_t, t_from, t_to=None):
    t_tensor = torch.tensor([t_from], device=x_t.device)
    model_output = model(x_t, t_tensor)
    eps_pred, _ = torch.split(model_output, x_t.shape[1], dim=1)

    alpha_t = alphas[t_from]
    beta_t = betas[t_from]
    alpha_bar_t = alphas_cumprod[t_from]

    mean = (1.0 / np.sqrt(alpha_t)) * (
        x_t - (beta_t / np.sqrt(1.0 - alpha_bar_t)) * eps_pred
    )

    if t_from == 0:
        return mean

    z = torch.randn_like(x_t)
    sigma_t = np.sqrt(beta_t)
    return mean + sigma_t * z

In [ ]:
probe_row, probe_col = 128, 128
probe_channel = 0
timesteps_to_save = [950, 700, 500, 300, 100, 50]

torch.manual_seed(42)
img = torch.randn(1, 3, 256, 256, device=device)

trajectory = {}
greens_trajectory = {}

schedule = list(range(999, -1, -1))
save_set = set(timesteps_to_save)

for i in tqdm(range(len(schedule) - 1), desc="Generating trajectory"):
    t_from = schedule[i]
    t_to = schedule[i + 1]

    if t_from in save_set:
        trajectory[t_from] = img.detach().clone()
        e_i = make_pixel_delta(img.shape, probe_channel, probe_row, probe_col, device=device)
        g = score_jvp(img, t_from, e_i)
        greens_trajectory[t_from] = g[0].cpu().numpy()

    with torch.no_grad():
        img = ddpm_step(img, t_from, t_to)

trajectory[0] = img.detach().clone()

Generating trajectory:   0%|          | 0/999 [00:00<?, ?it/s]

In [ ]:
t_list = sorted(greens_trajectory.keys(), reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

max_radius = 128
radii = np.arange(1, max_radius + 1)
yy, xx = np.mgrid[0:256, 0:256]
dist = np.sqrt((yy - probe_row) ** 2 + (xx - probe_col) ** 2)

r50_traj, r90_traj, norm_traj, t_traj = [], [], [], []
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(t_list)))

for j, t in enumerate(t_list):
    g = greens_trajectory[t].copy()
    g[:, probe_row, probe_col] = 0.0

    energy = np.sum(g**2, axis=0)
    total_energy = energy.sum()
    if total_energy < 1e-20:
        continue

    enclosed = np.array([energy[dist <= r].sum() / total_energy for r in radii])
    axes[0].plot(radii, enclosed, linewidth=2, color=colors[j], label=f"t={t}")

    idx50 = min(np.searchsorted(enclosed, 0.5), len(radii) - 1)
    idx90 = min(np.searchsorted(enclosed, 0.9), len(radii) - 1)

    r50_traj.append(radii[idx50])
    r90_traj.append(radii[idx90])
    norm_traj.append(np.linalg.norm(greens_trajectory[t]))
    t_traj.append(t)

axes[0].axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
axes[0].axhline(y=0.9, color='gray', linestyle='--', alpha=0.4)
axes[0].set_xlabel("Radius (pixels)")
axes[0].set_ylabel("Fraction of energy enclosed")
axes[0].set_title("Encircled energy along generation trajectory")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, max_radius)

ax2 = axes[1]
ax2_twin = ax2.twinx()
ln1 = ax2.plot(t_traj, r90_traj, 'o-', linewidth=2, markersize=8, label="90% radius")
ln2 = ax2.plot(t_traj, r50_traj, 's--', linewidth=2, markersize=8, label="50% radius")
ln3 = ax2_twin.plot(t_traj, norm_traj, 'D-', linewidth=2, markersize=8, label=r"$\|J_s e_i\|_2$")

ax2.set_xlabel("Timestep t")
ax2.set_ylabel("Effective radius (pixels)")
ax2_twin.set_ylabel(r"$\|J_s e_i\|_2$")
ax2.set_title("Receptive field evolution during generation")
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3)

lns = ln1 + ln2 + ln3
labs = [l.get_label() for l in lns]
ax2.legend(lns, labs, fontsize=10, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
timesteps_stability = sorted(list(range(950, 0, -50)) + [25, 10, 5], reverse=True)

torch.manual_seed(0)
z_stab = torch.randn_like(x_0).to(device)

results = []
num_pi_iters = 50

for t in tqdm(timesteps_stability, desc="Stability sweep"):
    ab = alphas_cumprod[t]
    x_t_curr = (np.sqrt(ab) * x_0 + np.sqrt(1.0 - ab) * z_stab).to(device)

    eig_full, rho_full, _ = power_iteration(x_t_curr, t, num_iters=num_pi_iters, use_full_step=True)
    eig_score, rho_score, _ = power_iteration(x_t_curr, t, num_iters=num_pi_iters, use_full_step=False)

    results.append({
        "t": t,
        "alpha_bar": ab,
        "alpha_t": alphas[t],
        "beta_t": betas[t],
        "snr_db": 10 * np.log10(ab / (1.0 - ab)),
        "eig_full": eig_full,
        "rho_full": rho_full,
        "eig_score": eig_score,
        "rho_score": rho_score,
    })

t_arr = np.array([r["t"] for r in results])
rho_full_arr = np.array([r["rho_full"] for r in results])
eig_full_arr = np.array([r["eig_full"] for r in results])
rho_score_arr = np.array([r["rho_score"] for r in results])
snr_arr = np.array([r["snr_db"] for r in results])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

ax = axes[0, 0]
ax.plot(t_arr, rho_full_arr, 'o-', linewidth=2, markersize=4, label=r'$\rho(J_f)$')
ax.axhline(y=1.0, color='r', linestyle='--', linewidth=2, alpha=0.7, label='Stability boundary')
ax.fill_between(t_arr, 1.0, rho_full_arr, where=(rho_full_arr > 1), color='r', alpha=0.15)
ax.fill_between(t_arr, rho_full_arr, 1.0, where=(rho_full_arr < 1), color='b', alpha=0.15)
ax.set_xlabel('Timestep t')
ax.set_ylabel(r'Spectral radius $\rho(J_f)$')
ax.set_title('Spectral radius of DDPM reverse step')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

ax = axes[0, 1]
ax.plot(t_arr, eig_full_arr, 'o-', linewidth=2, markersize=4, label=r'$\lambda_{\max}(J_f)$')
ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
ax.axhline(y=-1.0, color='r', linestyle='--', alpha=0.5)
ax.axhline(y=0.0, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('Timestep t')
ax.set_ylabel(r'Dominant eigenvalue')
ax.set_title('Signed dominant eigenvalue of $J_f$')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

ax = axes[1, 0]
ax.plot(t_arr, rho_score_arr, 'o-', linewidth=2, markersize=4, label=r'$\rho(J_s)$')
ax.set_xlabel('Timestep t')
ax.set_ylabel(r'Spectral radius $\rho(J_s)$')
ax.set_title('Spectral radius of score Jacobian alone')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

ax = axes[1, 1]
ax.plot(snr_arr, rho_full_arr, 'o-', linewidth=2, markersize=4, label=r'$\rho(J_f)$')
ax.axhline(y=1.0, color='r', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Spectral radius')
ax.set_title('Spectral radius vs. signal-to-noise ratio')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Von Neumann Stability Analysis of DDPM Reverse Process', y=1.02)
plt.tight_layout()
plt.show()